# Lecture 2 – Technical Agentic AI: Building Agents with Google Calendar

This notebook is the hands-on part of the **"Technical Agentic AI"** block of the Lecture 2 slides.
The slides explain the concepts; this notebook is where you build and run them.

**How to read this notebook**

| Marker | Meaning |
|---|---|
| 📖 | Short concept recap |
| 🧪 | Exercise: run the code and read what happens |
| ✍️ | **Your turn**: the task that you complete yourself |

### The 7 exercise sections

| Section | Title | Exercises | Slides |
|---|---|---|---|
| 1 | Setup and Initialization | 1.1, 1.2 | 37–38 |
| 2 | Function Tools and Your First Agent | 2.1, 2.2, 2.3 | 39–44 |
| 3 | Google Calendar Tools | 3.1, 3.2 | 37, 45 |
| 4 | The Calendar Scheduling Agent | 4.1, 4.2 | 46 |
| 5 | Example User Flow | 5.1 | 47 |
| 6 | Interactive Chat Interface | 6.1 | 47 |
| 7 | Agent-to-Agent Collaboration | 7.1, 7.2 | 48–51 |
| Bonus | Deterministic Flow (optional) | – | 52–53 |

### Learning objectives
By the end of this notebook you can:
1. Connect the OpenAI Agents SDK to Azure OpenAI (Section 1)
2. Turn Python functions into tools with `@function_tool` and give them to an agent (Section 2)
3. Wrap the Google Calendar API from Lecture 1 as agent tools (Section 3)
4. Build and test an agent that schedules real calendar events from natural language (Section 4)
5. Run a multi-step user flow and a chat interface that remembers the conversation (Sections 5–6)
6. Use one agent as a tool of another agent (Section 7)

### Prerequisites
- Lecture 1 finished: the Google Calendar API works on your machine (`token.pickle` or OAuth client in `.env`)
- Azure OpenAI credentials in `.env`: `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`, `OPENAI_VERSION_NAME`, `OPENAI_DEPLOYMENT_NAME`
- Google OAuth client in `.env`: `GOOGLE_CLIENT_ID`, `GOOGLE_CLIENT_SECRET`
- Basic understanding of `async` / `await` in Python

> ⚠️ This notebook creates **real events** in your Google Calendar. Use a test calendar or delete the events afterwards.

## 📖 Concepts Recap 

**SDK** A Software Development Kit gives you libraries, APIs, documentation and utilities so you don't reinvent the wheel.

**OpenAI Agents SDK core concepts**

| Concept | Meaning | Where in this notebook |
|---|---|---|
| **Agent** | An LLM with instructions, tools, guardrails and handoffs: a specialised worker with a job description | Exercises 2.2, 4.1, 7.1 |
| **Tool** | An external capability the agent can call. Three types: *hosted tools*, *function tools*, *agent-as-tool* | Function tools: 2.1, 3.2 · Agent-as-tool: 7.1 |
| **Guardrail** | Checks on the agent's input or output (e.g. "must be a scheduling question") | Not covered in the exercises |
| **Handoff** | A special tool call that transfers control to another agent, which then talks to the user | Not covered (Section 7 uses agent-as-tool instead) |
| **Session** | Conversation history kept across runs | Exercises 5.1 and 6.1 (conversation memory) |
| **Tracing** | Built-in tracking of agent runs | Disabled in Exercise 1.2 (we use Azure, not OpenAI's tracing backend) |
| **Runner** | Runs an agent: the loop that calls the model, executes tools and returns `final_output` | Every test cell |

**How does an agent choose a tool?** It compares the user query with each tool's *name and description* (semantic match),
checks whether it can fill in the parameters (input compatibility), and respects guardrails and availability.
👉 This is why the **docstring of every tool matters**.

**Agent-as-tool vs handoff.** With *agent-as-tool* (Section 7) the main agent calls a sub-agent, gets the answer back and stays in charge.
With a *handoff* the second agent takes over the conversation.

**MCP** The Model Context Protocol is an open standard for giving models structured context from external systems.
MCP is about *context* (passive); agents are about *actions* (active). We don't use MCP in this notebook.

## Section 1: Setup and Initialization 🔧

**Slides** First check that last lecture's Google Calendar setup still works, then connect the Agents SDK to Azure OpenAI.
In Lecture 1 we called the LLM directly; now the **Agents SDK** sits in between and handles tool calls for us.

### 🧪 Exercise 1.1: Import Libraries and Load the Environment

Install the packages once (uncomment the `%pip` line), then import everything we need.

> 💡 `openai-agents` needs a recent `openai` package (3.x). If your `requirements.txt` pins an old `openai` version, the import below will fail. Upgrade it.

In [ ]:
print("📦 Installing packages... (this will take a few minutes)")
!pip install -r requirements-lecture_2.txt
# %pip install -U openai openai-agents python-dotenv google-api-python-client google-auth-httplib2 google-auth-oauthlib
print("✅ Packages installed successfully!")

In [ ]:
# 📦 Exercise 1.1: Import libraries and load the environment
# Run once to install (installs into the active kernel):

import os
import datetime
import pickle
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# OpenAI Agents SDK
from agents import (
    Agent,
    Runner,
    function_tool,
    set_default_openai_client,
    set_tracing_disabled,
)
from openai import AsyncAzureOpenAI

# Google Calendar libraries (from Lecture 1)
from googleapiclient.discovery import build
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow

print("✅ All libraries imported successfully!")

### 🧪 Exercise 1.2: Configure Azure OpenAI for Agents

We create an async Azure client and make it the default client for every agent.
Tracing is switched off because it would try to send traces to OpenAI's platform with an OpenAI key we don't have.

In [ ]:
# 🔧 Exercise 1.2: Configure Azure OpenAI for agents
print("🔧 Configuring Azure OpenAI for agents...")

# Configure Azure OpenAI client (more advanced, and easier than Lecture 1)
azure_client = AsyncAzureOpenAI(
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    api_version="v1",
    base_url=os.environ.get("AZURE_OPENAI_ENDPOINT").rsplit("/", 1)[0] + "/",
)

# Every Agent will use this client by default
set_default_openai_client(azure_client, use_for_tracing=False)

# Turn off tracing (it needs an OpenAI platform key)
set_tracing_disabled(True)

# One place to choose the model deployment for all agents in this notebook
MODEL = os.environ.get("OPENAI_DEPLOYMENT_NAME", "gpt-5.6-luna")

missing = [v for v in ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT", "OPENAI_VERSION_NAME"] if not os.environ.get(v)]
if missing:
    print(f"⚠️ Missing in .env: {', '.join(missing)}")
else:
    print("✅ Azure OpenAI client configured for agents!")
    print(f"📝 Model deployment: {MODEL}")

## Section 2: Function Tools and Your First Agent 🛠️

The `@function_tool` decorator registers a normal Python function so the agent knows:
- **what** the function does (from the docstring),
- **which inputs** it accepts (from the type hints),
- **how to call it** automatically when a question needs it.

The SDK wraps your function with metadata (name, description, parameter schema). **The description is important:** it is how the agent decides which tool to use.

```python
from agents import function_tool

@function_tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """Fetch the current weather for a given city."""
    return f"The weather in {city} is 18° {unit}."
```

### 🧪 Exercise 2.1: Create Three Function Tools

We create three simple tools: get the current date, calculate a duration, and get (mock) weather.

> 💡 `get_current_date` also returns the **weekday**. Without it, the model has to work out the weekday itself to resolve "next Tuesday", and it often gets it wrong.

In [ ]:
# 🧪 Exercise 2.1: Create three function tools

@function_tool
async def get_current_date() -> str:
    """Get today's date in YYYY-MM-DD format together with the weekday name.
    Use this to resolve relative dates such as 'tomorrow' or 'next Tuesday'."""
    today = datetime.date.today()
    return f"{today.strftime('%Y-%m-%d')} ({today.strftime('%A')})"


@function_tool
async def calculate_duration(start_time: str, end_time: str) -> str:
    """
    Calculate the duration between two times.

    Args:
        start_time: Start time in HH:MM format (24-hour)
        end_time: End time in HH:MM format (24-hour)

    Returns:
        Duration as a string (e.g. "2 hours 30 minutes")
    """
    try:
        start_hour, start_min = map(int, start_time.split(":"))
        end_hour, end_min = map(int, end_time.split(":"))

        start_total = start_hour * 60 + start_min
        end_total = end_hour * 60 + end_min
        if end_total < start_total:  # ends the next day
            end_total += 24 * 60

        hours, minutes = divmod(end_total - start_total, 60)
        if hours and minutes:
            return f"{hours} hours {minutes} minutes"
        if hours:
            return f"{hours} hours"
        return f"{minutes} minutes"
    except Exception as e:
        return f"Error calculating duration: {e}"


@function_tool
async def get_weather_mock(city: str) -> str:
    """Get mock weather information for a city."""
    weather_data = {
        "brussels": "🌤️ Partly cloudy, 18°C",
        "london": "🌧️ Rainy, 15°C",
        "paris": "☀️ Sunny, 22°C",
        "new york": "🌥️ Cloudy, 20°C",
    }
    return weather_data.get(city.lower(), f"🌍 Weather data not available for {city}")


print("✅ Function tools created: get_current_date, calculate_duration, get_weather_mock")

### 🧪 Exercise 2.2: Create a Simple Agent That Chooses Between Tools

An agent gets a **name**, a **model**, **instructions** (its job description) and a list of **tools**.
It decides by itself which tool to call for a question about the date, a time duration or the weather.

#### Agent creation template
Use this as a reference whenever you build an agent:

```python
my_agent = Agent(
    name="[Agent Name]",          # Give your agent a clear name
    model=MODEL,                  # The model deployment
    instructions="""You are a [type of assistant] with access to [capabilities].

    Your primary tasks:
    - [Task 1]: [Description]
    - [Task 2]: [Description]

    Behavior guidelines:
    - When [situation], do [action]
    - Use tools when [condition]

    Communication style:
    - [Style instruction]""",
    tools=[function_1, function_2],   # List of @function_tool functions
)
```

In [ ]:
# 🧪 Exercise 2.2: Create a simple agent that chooses between tools
simple_agent = Agent(
    name="Helper Assistant",
    model=MODEL,
    instructions="""You are a helpful assistant with access to useful tools.

    Use the available tools to help users with:
    - Getting the current date (use get_current_date)
    - Calculating time durations (use calculate_duration)
    - Getting weather information (use get_weather_mock)

    Always use the tools when relevant and explain briefly what you did.""",
    tools=[get_current_date, calculate_duration, get_weather_mock],
)

print(f"✅ Agent '{simple_agent.name}' created with {len(simple_agent.tools)} tools")

### 🧪 Exercise 2.3: Test the Simple Agent

We use a reusable helper, `ask_agent_question`, to send a question to an agent with the `Runner`.
It also prints **which tools the agent called**, so you can see its choice.
We reuse this helper in Sections 4 and 7.

In [ ]:
# 🧪 Exercise 2.3 (part 1): Reusable helpers to question any agent

def show_tool_calls(result) -> list[str]:
    """Print and return the names of the tools the agent called during a run."""
    calls = [
        getattr(item.raw_item, "name", "unknown")
        for item in result.new_items
        if item.type == "tool_call_item"
    ]
    print(f"🔧 Tools called: {', '.join(calls) if calls else '(none, answered directly)'}")
    return calls


async def ask_agent_question(agent: Agent, question: str, max_turns: int = 5):
    """
    Ask any agent a question and print the answer.

    Args:
        agent: The agent to question
        question: The question or request
        max_turns: Maximum number of model turns (each tool call uses one turn)

    Returns:
        The RunResult (use result.final_output for the answer text)
    """
    print(f"👤 User: {question}")
    print("-" * 60)
    try:
        result = await Runner().run(starting_agent=agent, input=question, max_turns=max_turns)
        show_tool_calls(result)
        print(f"🤖 {agent.name}: {result.final_output}")
        return result
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        print("=" * 60 + "\n")

print("✅ Helpers ready: ask_agent_question(), show_tool_calls()")

In [ ]:
# 🧪 Exercise 2.3 (part 2): Test scenarios for the simple agent
simple_test_questions = [
    "What's today's date?",                          # → get_current_date
    "How long is a meeting from 14:30 to 16:45?",    # → calculate_duration
    "What's the weather like in Brussels?",          # → get_weather_mock
]

for question in simple_test_questions:
    await ask_agent_question(simple_agent, question)

#### ✍️ Your turn
Ask your own question about a **time duration**. Try a 12-hour format ("from 9 AM to 1:15 PM") or a meeting that ends after midnight.

In [ ]:
# ✍️ Your turn: ask your own question about a time duration
my_question = "How long is a shift from 10 PM to 6:30 AM?"   # ← change this

await ask_agent_question(simple_agent, my_question)

## Section 3: Google Calendar Tools 📅

We reuse the Google Calendar functions from Lecture 1 and turn them into function tools the agent can call.

Key points:
- **Authentication:** reuse your existing credentials
- **Function tool conversion:** add `@function_tool` and a clear docstring
- **Error handling:** return an error *message* instead of raising, so the agent can explain the problem to the user
- **Clear return values:** the agent reads the returned string to write its answer

### 🧪 Exercise 3.1: Connect to Google Calendar (from Lecture 1)

**"Did everyone make the Google Calendar API work?"** This is the same authentication as last lecture.

In [ ]:
# 🔧 Exercise 3.1: Connect to Google Calendar (from Lecture 1)
GOOGLE_CLIENT_ID = os.getenv("GOOGLE_CLIENT_ID")
GOOGLE_CLIENT_SECRET = os.getenv("GOOGLE_CLIENT_SECRET")

# If you change these scopes, delete token.pickle and log in again.
SCOPES = ["https://www.googleapis.com/auth/calendar"]


def get_calendar_service():
    """Return an authenticated Google Calendar service (reuses token.pickle)."""
    creds = None
    if os.path.exists("token.pickle"):
        with open("token.pickle", "rb") as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_config(
                {
                    "installed": {
                        "client_id": GOOGLE_CLIENT_ID,
                        "client_secret": GOOGLE_CLIENT_SECRET,
                        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
                        "token_uri": "https://oauth2.googleapis.com/token",
                    }
                },
                SCOPES,
            )
            creds = flow.run_local_server(port=0)  # opens the browser and handles the redirect

        with open("token.pickle", "wb") as token:
            pickle.dump(creds, token)

    return build("calendar", "v3", credentials=creds)


# Quick check that the connection works
try:
    get_calendar_service().calendarList().get(calendarId="primary").execute()
    print("✅ Connected to Google Calendar!")
except Exception as e:
    print(f"❌ Could not connect to Google Calendar: {e}")
    print("👉 Revisit the Lecture 1 setup before continuing.")

### 🧪 Exercise 3.2: Turn the Calendar Functions into Function Tools

`create_calendar_event` from last lecture becomes a function tool (we did this for you).
We also add a second tool, `list_upcoming_events`, so the agent can read your agenda and spot conflicts.

In [ ]:
# 🛠️ Exercise 3.2: Calendar function tools

@function_tool
async def create_calendar_event(
    title: str,
    date: str,
    start_time: str,
    end_time: str,
    location: str = "",
    description: str = "",
) -> str:
    """
    Create a Google Calendar event.

    Args:
        title: Event title (required)
        date: Event date in YYYY-MM-DD format (required)
        start_time: Start time in HH:MM format, 24-hour (required)
        end_time: End time in HH:MM format, 24-hour (required)
        location: Event location (optional)
        description: Event description (optional)

    Returns:
        Success message with the event details and link, or an error message
    """
    try:
        service = get_calendar_service()

        event = {
            "summary": title,
            "location": location,
            "description": description or f"Event: {title}",
            "start": {"dateTime": f"{date}T{start_time}:00", "timeZone": "Europe/Brussels"},
            "end": {"dateTime": f"{date}T{end_time}:00", "timeZone": "Europe/Brussels"},
            "reminders": {
                "useDefault": False,
                "overrides": [
                    {"method": "popup", "minutes": 15},
                    {"method": "email", "minutes": 60},
                ],
            },
        }

        created_event = service.events().insert(calendarId="primary", body=event).execute()
        event_link = created_event.get("htmlLink", "No link available")
        return f"✅ Created event '{title}' on {date} from {start_time} to {end_time}. Link: {event_link}"

    except Exception as e:
        return f"❌ Failed to create event: {e}"


@function_tool
async def list_upcoming_events(max_results: int = 5) -> str:
    """
    List upcoming events from Google Calendar, starting from now.
    Use this to show the user's agenda or to check for scheduling conflicts.

    Args:
        max_results: Maximum number of events to return (default: 5)

    Returns:
        A numbered list of upcoming events with date, time and location
    """
    try:
        service = get_calendar_service()
        now = datetime.datetime.now(datetime.timezone.utc).isoformat()

        events = service.events().list(
            calendarId="primary",
            timeMin=now,
            maxResults=max_results,
            singleEvents=True,
            orderBy="startTime",
        ).execute().get("items", [])

        if not events:
            return "📅 No upcoming events found."

        lines = [f"📅 Next {len(events)} upcoming events:"]
        for i, event in enumerate(events, 1):
            start = event["start"].get("dateTime", event["start"].get("date"))
            title = event.get("summary", "No title")
            location = event.get("location", "")

            if "T" in start:  # timed event
                start_dt = datetime.datetime.fromisoformat(start.replace("Z", "+00:00"))
                when = start_dt.strftime("%Y-%m-%d (%A) at %H:%M")
            else:             # all-day event
                when = f"{start} (all day)"

            line = f"{i}. {title} - {when}"
            if location:
                line += f" at {location}"
            lines.append(line)

        return "\n".join(lines)

    except Exception as e:
        return f"❌ Failed to list events: {e}"


print("✅ Calendar tools created: create_calendar_event, list_upcoming_events")

## Section 4: The Calendar Scheduling Agent 🤖

**Slide 46.** We now combine **4 tools** into one agent that:
- schedules an event (today, tomorrow, next week …),
- lists upcoming events and checks for conflicts,
- calculates durations for time planning.

Agent design principles:
1. **Clear instructions:** tell the agent exactly what to do and in which format
2. **The right tools:** give it only what it needs
3. **Error handling:** tell it what to do when information is missing or a tool fails
4. **User experience:** confirm details and give clear feedback

### 🧪 Exercise 4.1: Create the Calendar Agent

In [ ]:
# 🤖 Exercise 4.1: Create the calendar agent (4 tools)
calendar_agent = Agent(
    name="Calendar Scheduler Pro",
    model=MODEL,
    instructions="""You are an expert calendar scheduling assistant with access to Google Calendar.

Your primary job is to help users schedule events efficiently and accurately.

WHEN SCHEDULING EVENTS:
1. Extract the key details: title, date, start time, end time, location, description
2. If ANY required detail is missing (title, date, start_time, end_time), ask the user for it
3. Use create_calendar_event to schedule the event
4. Confirm successful scheduling with the event details and link

FORMATTING REQUIREMENTS:
- Dates: YYYY-MM-DD (e.g. 2025-10-29)
- Times: 24-hour HH:MM (e.g. 14:30 for 2:30 PM); convert 12-hour times

HELPFUL TOOLS:
- get_current_date: resolve relative dates like "tomorrow" or "next Monday" (it returns the weekday too)
- list_upcoming_events: show the agenda and check for conflicts
- calculate_duration: help with time planning

COMMUNICATION STYLE:
- Friendly and professional
- Clear success messages with event links
- Ask clarifying questions when information is incomplete

Remember: you create REAL calendar events.""",
    tools=[
        create_calendar_event,
        list_upcoming_events,
        get_current_date,
        calculate_duration,
    ],
)

print(f"✅ Agent '{calendar_agent.name}' created with {len(calendar_agent.tools)} tools")

### 🧪 Exercise 4.2: Test the Calendar Agent

Three test scenarios: a simple event with a relative date, a detailed event, and a question about your agenda.
Check the **🔧 Tools called** line: does the agent call `get_current_date` before it schedules "tomorrow"?

In [ ]:
# 🧪 Exercise 4.2: Test scenarios for the calendar agent
workshop_date = (datetime.date.today() + datetime.timedelta(days=7)).isoformat()   # always in the future

calendar_test_scenarios = [
    {
        "name": "Simple event (relative date)",
        "input": "Schedule a team meeting tomorrow from 2:00 PM to 3:30 PM in Conference Room A",
    },
    {
        "name": "Detailed event",
        "input": f"Create an event for 'Machine Learning Workshop' on {workshop_date} from 09:00 to 12:00 "
                 "at University Campus, Room 101. This is a hands-on workshop about neural networks.",
    },
    {
        "name": "Check upcoming events",
        "input": "What are my upcoming events this week?",
    },
]

for scenario in calendar_test_scenarios:
    print(f"📝 Test: {scenario['name']}")
    await ask_agent_question(calendar_agent, scenario["input"])

#### ✍️ Your turn
Ask the agent to **schedule a lecture this week** in your agenda. Then open Google Calendar and check that the date and time are correct.

In [ ]:
# ✍️ Your turn: schedule a lecture this week
my_request = "Schedule an 'Applied AI lecture' this Thursday from 1:30 PM to 4:30 PM in room B.101"   # ← change this

await ask_agent_question(calendar_agent, my_request)

## Section 5: Example User Flow 🔁

A user rarely asks one isolated question. In this flow the user looks at their agenda, schedules something,
and then asks a **follow-up** that only makes sense if the agent remembers the earlier turns.

📖 **Conversation memory.** Each `Runner.run` call starts with no memory. To continue a conversation we pass the previous turns back in:
`result.to_input_list()` returns the whole conversation so far, and we add the next user message to it.
This is what **Sessions** do for you automatically.

### 🧪 Exercise 5.1: Run a Multi-Step User Flow

In [ ]:
# 🔁 Exercise 5.1: Example user flow (multi-turn, with memory)
user_flow = [
    "What are my upcoming events?",
    "Schedule a Python study session tomorrow from 7 PM to 9 PM at home",
    "Does that study session overlap with any of the events you listed before?",   # needs memory!
]

conversation = []   # the conversation so far, passed back to the agent on every turn

for step, user_input in enumerate(user_flow, 1):
    print(f"👤 Step {step}: {user_input}")
    try:
        result = await Runner().run(
            starting_agent=calendar_agent,
            input=conversation + [{"role": "user", "content": user_input}],
            max_turns=5,
        )
        conversation = result.to_input_list()   # remember everything for the next step
        show_tool_calls(result)
        print(f"🤖 Assistant: {result.final_output}")
    except Exception as e:
        print(f"❌ Error: {e}")
    print("-" * 60 + "\n")

print("✅ User flow complete! Edit 'user_flow' to try your own sequence.")

## Section 6: Interactive Chat Interface 💬

Now *you* are the user. The chat loop below reads your input, sends it with the conversation history to the
calendar agent, and prints the answer. In VS Code an input box opens at the top of the window.

Type `quit`, `exit`, `bye` or `q` to stop.

### 🧪 Exercise 6.1: Build the Chat Interface

In [ ]:
# 💬 Exercise 6.1 (part 1): Define the chat loop
async def chat_with_agent(agent: Agent = calendar_agent):
    """Chat with an agent in the notebook. The agent remembers the whole conversation."""
    conversation = []
    print(f"💬 Chatting with '{agent.name}'. Type 'quit' to stop.\n")

    while True:
        try:
            user_input = input("You: ").strip()

            if user_input.lower() in ["quit", "exit", "bye", "q"]:
                print("\n🤖 Calendar Agent: Goodbye! Have a great day! 👋")
                break
            if not user_input:
                continue

            print(f"\n👤 You: {user_input}")
            result = await Runner().run(
                starting_agent=agent,
                input=conversation + [{"role": "user", "content": user_input}],
                max_turns=5,
            )
            conversation = result.to_input_list()   # keep the memory
            print(f"🤖 Calendar Agent: {result.final_output}")
            print("\n" + "-" * 50 + "\n")

        except KeyboardInterrupt:
            print("\n\n🤖 Calendar Agent: Chat interrupted. Goodbye! 👋")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")
            print("Please try again or type 'quit' to exit.\n")

print("✅ Chat interface ready. Run the next cell to start chatting.")

#### ✍️ Your turn
Start the chat and ask the interface something about **your schedule**, for example *"What do I have on Friday?"*
followed by *"Book a 1-hour gym session that day after my last event."* (the second question only works because the agent remembers the first).

In [ ]:
# ✍️ Your turn: start the chat (type 'quit' to stop)
await chat_with_agent()

## Section 7: Agent-to-Agent Collaboration 🤝

An agent can be used **as a tool** by another agent with `.as_tool()`.
The main agent (coordinator) decides when to call the sub-agent, receives its answer and stays in charge of the conversation.

How it works:
1. **Coordinator agent:** the calendar agent that talks to the user
2. **Sub-agent tool:** `simple_agent.as_tool(tool_name=..., tool_description=...)`
3. **Delegation:** the coordinator calls the sub-agent for date and duration questions
4. **Integration:** the sub-agent's answer is used in the final response

Why this is useful:
- **Specialisation:** each agent is good at one thing
- **Reusability:** the same sub-agent can serve several coordinators
- **Modularity:** change one agent without rebuilding the others

> 💡 **Instructions are again key.** The coordinator only sees the tool *name* and *description*.
> Refer to the sub-agent by its tool name (`date_calculator_time_helper`) in the instructions, or the coordinator won't connect the two.

For a full example, see `examples/agent_patterns/agents_as_tools_conditional.py` in the openai-agents-python GitHub repository.

### 🧪 Exercise 7.1: Embed the Simple Agent in a New Calendar Agent

Notice that this agent has **no** `get_current_date` or `calculate_duration` tools. It must ask the sub-agent instead.

In [ ]:
# 🤖 Exercise 7.1: Advanced calendar agent that uses the simple agent as a tool
date_helper_tool = simple_agent.as_tool(
    tool_name="date_calculator_time_helper",
    tool_description=(
        "Expert assistant for today's date, relative date resolution (tomorrow, next week, next Friday), "
        "time duration calculations and time arithmetic. Use it for any date or time question."
    ),
)

calendar_agent_to_agent = Agent(
    name="Calendar Scheduler Pro Max",
    model=MODEL,
    instructions="""You are an advanced calendar scheduling assistant with access to Google Calendar
and a specialised helper agent.

SCHEDULING WORKFLOW:
1. Parse the request for: title, date, start_time, end_time, location, description
2. If required information is missing, ask specific clarifying questions
3. Use the date_calculator_time_helper tool for ALL date resolution (e.g. "tomorrow", "next Friday")
   and duration or end-time calculations. Do not calculate dates yourself.
4. Use list_upcoming_events to check for conflicts when the user asks or when the slot may be busy
5. Create events with create_calendar_event
6. Confirm the result with all event details and the event link

DATE & TIME FORMAT:
- Dates as YYYY-MM-DD, times as 24-hour HH:MM
- Time zone: Europe/Brussels
- Explain any conversion you make (e.g. "next Tuesday = 2025-11-18")

ERROR HANDLING:
- If event creation fails, explain why
- If there is a conflict, suggest an alternative time
- If the request is vague (e.g. "sometime Friday morning"), propose a time and ask for confirmation

COMMUNICATION STYLE:
- Professional, friendly and clear; use emojis sparingly""",
    tools=[
        create_calendar_event,
        list_upcoming_events,
        date_helper_tool,
    ],
)

print(f"✅ Agent '{calendar_agent_to_agent.name}' created")
print("🔧 Tools:", [tool.name for tool in calendar_agent_to_agent.tools])

### 🧪 Exercise 7.2: Test, Test, Test

We run five scenarios. For each one we check:
- did the run finish without errors?
- did the coordinator **delegate** to `date_calculator_time_helper`?

A run that finishes is not automatically correct. **Always check the dates in your calendar.**

In [ ]:
# 🧪 Exercise 7.2: Test scenarios for agent collaboration
advanced_test_scenarios = [
    {
        "name": "Relative date with duration calculation",
        "input": "Schedule a 'Project Planning Session' next Tuesday from 2 PM to 5:30 PM in the main "
                 "conference room. I need to know how long this meeting will be.",
        "expected": "Delegates next Tuesday's date and the duration to the helper",
    },
    {
        "name": "Scheduling with conflict check",
        "input": "I want to schedule a 'Team Retrospective' tomorrow afternoon for 2 hours starting at 3 PM. "
                 "Can you check if I have any conflicts and create the event?",
        "expected": "Calls list_upcoming_events and lets the helper compute the end time",
    },
    {
        "name": "Multi-event request",
        "input": "Schedule three 30-minute interviews next week: Monday at 10 AM, Wednesday at 2 PM, "
                 "and Friday at 9 AM. All in Room B-202.",
        "expected": "Resolves three dates and creates three events",
    },
    {
        "name": "Natural language time",
        "input": "Book a 'Lunch Meeting with Sarah' the day after tomorrow from noon until 1:30 PM "
                 "at the Italian restaurant downtown",
        "expected": "Converts the relative date and 12-hour times",
    },
    {
        "name": "Vague request",
        "input": "I need a 90-minute workshop called 'AI Ethics Discussion' sometime next Friday morning. "
                 "Can you suggest a good time and create it?",
        "expected": "Proposes a time and calculates the end time",
    },
]


async def run_advanced_test_scenarios():
    """Run all scenarios and report errors and delegation."""
    report = []
    for i, scenario in enumerate(advanced_test_scenarios, 1):
        print(f"📝 Test {i}: {scenario['name']}")
        print(f"🎯 Expected: {scenario['expected']}")
        result = await ask_agent_question(calendar_agent_to_agent, scenario["input"])

        calls = [] if result is None else [
            getattr(item.raw_item, "name", "") for item in result.new_items if item.type == "tool_call_item"
        ]
        report.append({
            "scenario": scenario["name"],
            "finished": result is not None,
            "delegated": "date_calculator_time_helper" in calls,
        })

    print("📊 TEST SUMMARY")
    for r in report:
        print(f"  {'✅' if r['finished'] else '❌'} finished   "
              f"{'🤝' if r['delegated'] else '➖'} delegated   {r['scenario']}")
    print("\n👉 Now open Google Calendar and check that every date and weekday is correct.")
    return report


test_report = await run_advanced_test_scenarios()

#### ✍️ Your turn
Ask the advanced agent a question you **already asked the Section 4 agent**. The same answer should appear.
Run both agents below and compare the answers and the tools they used.

In [ ]:
# ✍️ Your turn: same question to both agents, compare the answers
repeat_question = "How long is a meeting from 14:30 to 16:45, and what is the date tomorrow?"   # ← change this

print("🅰️ Section 4 agent (own tools)")
await ask_agent_question(calendar_agent, repeat_question)

print("🅱️ Section 7 agent (delegates to the simple agent)")
await ask_agent_question(calendar_agent_to_agent, repeat_question)

## Bonus: Deterministic Flow (optional)

So far the agent chose **by itself** which tool to call: a **non-deterministic** flow.
In a **deterministic** flow *you* decide the order of steps in code, and you can add a **gate** that stops the flow when a step fails.

Useful when:
- several agents with different tools must run in a fixed order,
- one agent must not start unless the previous step succeeded.

Below: (1) the simple agent resolves a date → (2) a code gate checks the format → (3) the calendar agent checks for conflicts (it does **not** create anything).
See `examples/agent_patterns/deterministic.py` in the openai-agents-python repository for the full example.

In [ ]:
# 🔀 Bonus: deterministic flow with a gate
import re

async def deterministic_conflict_check(date_phrase: str, start_time: str, end_time: str):
    # Step 1: resolve the relative date with the simple agent
    step1 = await Runner().run(
        starting_agent=simple_agent,
        input=f"What is the date of '{date_phrase}'? Reply with ONLY the date in YYYY-MM-DD format.",
        max_turns=3,
    )
    date = step1.final_output.strip()
    print(f"1️⃣ Resolved '{date_phrase}' → {date}")

    # Step 2: gate. Stop if the output is not a valid date.
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", date):
        print("⛔ Gate: the date is not in YYYY-MM-DD format, so we stop here.")
        return None
    print("2️⃣ Gate passed: valid date format")

    # Step 3: conflict check with the calendar agent (no event creation)
    step3 = await Runner().run(
        starting_agent=calendar_agent,
        input=(f"Check my upcoming events (use max_results=20) for anything on {date} between "
               f"{start_time} and {end_time}. Do NOT create any event. Start your answer with "
               f"FREE or CONFLICT, followed by a one-line explanation."),
        max_turns=5,
    )
    print(f"3️⃣ {step3.final_output}")
    return step3.final_output

await deterministic_conflict_check("next Wednesday", "14:00", "16:00")

## Wrap-up: Where Does This Fit into Your Assignment? 🎓

Your assignment doesn't have to use the OpenAI Agents SDK everywhere, but the patterns from this notebook map directly onto it:

| Assignment agent (slide 57) | Built from |
|---|---|
| Parsing Agent: extract the schedule from a PDF or image | Lecture 1 (can be a one-time run per semester) |
| Calendar Agent: create and update events through the API | Sections 3–4 |
| Change Management Agent: "move soccer practice to Tuesday" | Sections 5–6 (conversation with memory) |
| Conflict Evaluation Agent: detect overlaps | `list_upcoming_events` + Section 4.2 / Bonus |
| Collaboration Agent(s): coordinate several agents | Section 7 (agent-as-tool) + Bonus (deterministic flow) |

### Recap
✅ Configured the Agents SDK with Azure OpenAI
✅ Created function tools and saw how the agent chooses between them
✅ Wrapped Google Calendar as tools and built a scheduling agent
✅ Ran a multi-step user flow and a chat with memory
✅ Used one agent as a tool of another agent